# Multi-Depot EVRPTW

**What is this variant?** Several depots serve the same set of customers. Routes can start/end at different warehouses. The benchmark still uses **one shared road graph** per city.

**Inputs you set below**
- **City graph** — same meaning as classic: `G` is the drivable network.
- **Primary depot** — `depot_lat`, `depot_lon` (first depot; internal id 0).
- **Additional depots** — extra `(lat, lon)` pairs you list manually (or your app could generate them).

**Customers:** either generated (`num_customers`) or loaded from **`customer_csv_path`** (then `num_customers` is `None` in config).

**Pipeline in one cell:** prepare graph + all depots → generate or load customers → choose stations → **finalize** into `instance`. **`generation_context`** holds intermediate data between those steps.

In [ ]:
# Prefer: pip install -e . (repo root) in this kernel. Fallback if import fails:
import sys
from pathlib import Path
for _p in (Path.cwd() / "src", Path.cwd().parent / "src"):
    if _p.is_dir() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

# Use the project environment; from the repo root run: pip install -e .
import evrp_instance_generator_framework as evrp
from evrp_instance_generator_framework.export.instance_export import export_instance
# Folium helpers (import from submodule so notebooks work even if the top-level namespace is stale)
from evrp_instance_generator_framework.visualization import (
    map_benchmark_interactive,
    map_city_roads_interactive,
    map_city_roads_with_depot_interactive,
)
# Variant-specific pipeline (multi-depot): prepare_graph_and_depots, generate_customers, ...
from evrp_instance_generator_framework.variants import multi_depot as md_variant

print("Using package from:", evrp.__file__)
print("Interactive map helpers loaded:", map_city_roads_interactive.__name__)


In [ ]:
# Reliable map rendering in most notebook frontends:
%matplotlib inline

# Optional interactive mode (pan/zoom) if supported by your frontend:
# %pip install ipympl
# %matplotlib widget

## City and road network

`prepare_city_road_network` downloads or loads OSM roads and returns **`G`**. Every depot and customer stop will be snapped to nodes of this graph—keep the same `G` for the whole notebook.

In [ ]:
city = "Casablanca"
country = "Morocco"
G = evrp.prepare_city_road_network(city, country, flat_terrain=True)
map_city_roads_interactive(G)


## Depots

- **`depot_lat`, `depot_lon`:** primary warehouse (facility coordinates).
- **`additional_depots`:** list of extra `(latitude, longitude)` pairs for other warehouses.

Each facility is snapped to the nearest road vertex within the configured distance. Customer generation uses **reachability from at least one depot**.

The plot below only highlights the primary depot pin; all depots are handled in the generation step.

In [ ]:
depot_lat = 33.573
depot_lon = -7.590
additional_depots = [(33.588, -7.612), (33.558, -7.565)]
map_city_roads_with_depot_interactive(G, depot_lat, depot_lon)


## Customer count (only if not using CSV)

Ignored when `customer_csv_path` points to a file; otherwise this is how many synthetic customers are created.

In [ ]:
num_customers = 16


## Charging stations count

How many charging facilities to **select** into the instance after customers exist. Same idea as classic: stations are placed with respect to demand and depots.

In [ ]:
num_stations = 5


## EV / vehicle parameters

Used for energy feasibility and metadata. Same `EVFeatures` object as in other variants.

In [ ]:
ev_features = evrp.EVFeatures(battery_capacity_kwh=75.0, driver_behavior="passive")


## Optional customer CSV

Same rules as classic: either automatic generation **or** a CSV path in `GenerationConfig`. CSV rows must be compatible with **this** graph (correct `movement_node_id` for `G`).

In [ ]:
customer_csv_path = None  # example: "my_customers.csv"


## Run generation (multi-depot pipeline)

Steps inside the code cell:

1. **`prepare_graph_and_depots`** — builds `generation_context`: graph, all depot records, merged travel-time views for customer filtering.
2. **`generate_customers`** — synthetic customers **or** load from `customer_csv_path` in config.
3. **`generate_stations`** — picks charging stations.
4. **`finalize`** — produces **`instance`** with metadata, feasibility, and everything needed for export/plots.

`compute_matrices=False` keeps this notebook fast; turn on if you need full matrices.

In [ ]:
if "customer_csv_path" not in globals():
    customer_csv_path = None
config = evrp.GenerationConfig(
    variant="multi_depot_evrptw",
    city=city,
    country=country,
    depot_lat=depot_lat,
    depot_lon=depot_lon,
    additional_depots=tuple(additional_depots),
    seed=42,
    num_customers=num_customers if customer_csv_path is None else None,
    customer_csv_path=customer_csv_path,
    num_stations=num_stations,
    customer_pattern="rc",
    time_window_tightness="medium",
)
generation_context = md_variant.prepare_graph_and_depots(config, ev_features, movement_graph=G)
generation_context = md_variant.generate_customers(generation_context)
generation_context = md_variant.generate_stations(generation_context)
# compute_matrices=False for speed; set True if you need distance/time matrices on instance
instance = md_variant.finalize(generation_context, compute_matrices=False, run_energy_feasibility=True)
map_benchmark_interactive(instance)


When using CSV, columns must match (names and types):  
`id`, `lat`, `lon`, `movement_node_id`, `snap_distance_m`, `demand`, `service_time_s`, `parking_time_s`, `time_open_s`, `time_close_s`.

Tip: export a generated instance once and reuse its customer table as a template.

## Export

Writes benchmark files (JSON here) under a folder name you choose. Inspect `instance` in Python or open the exported files in other tools.

In [ ]:
out_dir = export_instance(instance, output_dir="benchmark_export_multi_depot", fmt="json")
print("Exported to:", out_dir)
